# Optimization

Find the Pareto-optimal combinations of scenarios across all the stands of a project. The goal is to find the one scenario picked per stand, chosen so that no other combination is better in every target variable at once.

Stand areas come from the project's own `inputs/stand_data.json`. If it carries no area for one of the run's stands, this notebook raises rather than assuming equal areas.

In [ ]:
%matplotlib inline
import analysis.optimization.core as opti_core
import susi.io.load_output_data as load_output
import susi.io.utils as io_utils
from analysis.notebooks.components import (
    folder_selection,
    optimization,
    variable_selection,
)
from susi.io.app_settings import AppSettings

## Project and run selection
The optimization works over the folder holding one subfolder per stand, which is a *run* of a project (`projects/<project>/outputs/<run_id>/`), not the project folder itself. Narrow down to it as follows:

**projects root** (change only if your projects live outside the default root)

|

**project**

|

**run**

In [ ]:
folder_browser = folder_selection.build_folder_browser(AppSettings().projects_root)

In [ ]:
data_folder = folder_selection.resolve_start_folder(
    folder_browser, default=AppSettings().projects_root
)
project_dropdown = folder_selection.build_dropdown(data_folder, label="project")

In [ ]:
run_dropdown = folder_selection.build_run_dropdown(project_dropdown.value)

## Stand areas
Every stand is weighted by its area, so that larger stands count for more when the stands are combined into project-level totals.

In [ ]:
stand_areas_ha = optimization.display_stand_areas(
    project_dir=project_dropdown.value, run_id=run_dropdown.value.name
)

## Choose target variables
Tick the netcdf variables to optimize over. Three are preselected as a starting point; edit `DEFAULT_VARIABLES` below to change them.

In [ ]:
# The golden test netcdf is used only to read the variable structure of the
# netcdf file (names/shapes/units), not its values.
sample_netcdf_filepath = (
    io_utils.repo_root() / "tests/golden_file_test/golden_susi.nc"
)
all_variables = load_output.list_all_netcdf_variables(sample_netcdf_filepath)

DEFAULT_VARIABLES = [
    load_output.NetcdfVariablePath("/stand/volume"),
    load_output.NetcdfVariablePath("/balance/C/soil_c_balance_co2eq"),
    load_output.NetcdfVariablePath("/balance/N/to_water"),
]

selector = variable_selection.build_selector(
    all_variables, preselected=DEFAULT_VARIABLES
)

## Configure the targets
For each chosen variable, pick how its time/space array collapses to a single number, and whether its sign should be flipped. The optimizer always **minimizes**, so tick *Invert sign?* to maximize a variable instead (or to minimize one whose values are negative).

`Epsilon` controls the precision of the Pareto front: smaller is more precise but slower.

In [ ]:
chosen_netcdf_variables = variable_selection.checked_variables(selector, all_variables)

if not chosen_netcdf_variables:
    print("No variables chosen")
else:
    config = optimization.build_target_config(list(chosen_netcdf_variables.keys()))

## Read and pre-prune the data
Reads every stand's netcdf files and reduces them to one area-weighted number per scenario and target variable, then drops the scenarios that are already beaten within their own stand.

The printout says how big the remaining search is. Check it before running the next cell, which is the expensive one.

In [ ]:
variable_info = optimization.target_variable_properties(config)

prepared = opti_core.prepare_optimization_data(
    variable_info=variable_info,
    # `project_dirpath` is this parameter's name, but it takes a *run*
    # folder: the one holding a subfolder per stand.
    project_dirpath=run_dropdown.value,
    stand_areas=stand_areas_ha,
)

## Find the Pareto front
The dynamic-programming search itself. This is the slow step, and its cost grows with the number of combinations printed above and with how small `Epsilon` is.

`n_random_points` are sampled alongside it, to show in the plots what an arbitrary (non-optimal) combination looks like.

In [ ]:
results = opti_core.solve_optimization(
    prepared=prepared,
    epsilon=config.epsilon.value,
    n_random_points=10000,
)

## Results
Each panel plots one pair of target variables: the Pareto front against the random combinations.

In [ ]:
figure = optimization.display_pareto_corner_plot(
    results, labels=list(variable_info.keys())
)